---
title: "Tic-Tac-Toe Gameplay with RewriteGames"
format:
  html:
    theme: cosmo
    code-fold: false
    toc: true
jupyter: julia-1.11
execute:
  output: false
---

## Setup

In [ ]:
using Pkg
Pkg.activate("..")

using Catlab, AlgebraicRewriting
using RewriteGames
using Random
using Statistics: mean
using Catlab.Graphics.Graphviz: Attributes, Statement, Node
import Catlab.Graphics.Graphviz

---

## Overview

**RewriteGames.jl** models game states as *attributed C-sets* (ACSets) and
moves as *algebraic rewrite rules*.  Both concepts come from applied category
theory: ACSets are relational data structures whose schemas describe
objects (tables) and morphisms (foreign keys), and rewrite rules describe how
those structures transform under play.

This choice of representation is deliberately general.  Any game whose state
can be described as a relational structure — boards, graphs, maps, unit
networks — fits naturally into the framework.  Because every game shares the
same substrate, the engine's bookkeeping (match enumeration, budget tracking,
serialisation) applies uniformly across games without modification.

Two longer-term goals shape the library's design, though neither is fully
realised yet:

- **Modular game composition.**  ACSets and rewrite rules support algebraic
  composition: schemas can be combined, rules can be glued together, and
  schedules can be nested.  The infrastructure for this — `GameMigration`,
  compositional schedules — is in place.
- **Cross-game transfer.**  Because games share a common representation, a
  policy trained on one game could, in principle, transfer to another.  This is ongoing research; the current library provides the representational foundation but not yet the transfer machinery.

In this tutorial we use **tic-tac-toe** to demonstrate every layer of the
framework: simple enough to understand at a glance, rich enough to exercise
the full workflow.

1. Model the board as an ACSet and legal moves as rewrite rules.
2. Build wiring-diagram schedules that express turn order and win detection.
3. Use a *data migration functor* to derive O's schedule automatically from X's — avoiding
   duplicated code and guaranteeing structural symmetry.
4. Run self-play episodes and inspect the final board state.

---

## Part 1 — Modelling the Board as an ACSet

### Schema design

An ACSet's *schema* declares the tables (objects) and foreign keys (morphisms) that make
up a world state.  For tic-tac-toe we need:

| Object | Meaning |
|--------|---------|
| `Sq`   | Board squares (always exactly 9) |
| `E`    | Edges encoding the board's graph topology |
| `X`    | X pieces placed so far |
| `O`    | O pieces placed so far |

Two foreign keys record which square each piece occupies, and two more record the source
and target squares of each edge:

In [ ]:
#| label: schema
#| output: true
@present SchTTT(FreeSchema) begin
    Sq::Ob; E::Ob; X::Ob; O::Ob
    xsq::Hom(X, Sq); osq::Hom(O, Sq)
    src::Hom(E, Sq); tgt::Hom(E, Sq)
    SquareNum::AttrType
    num::Attr(Sq, SquareNum)
end

@acset_type TicTacToe(SchTTT, index=[:xsq, :osq])
const TTT = TicTacToe{Int}
𝒞 = ACSetCategory(VarACSetCat(TTT()))

to_graphviz(SchTTT; prog="dot")

We index on `:xsq` and `:osq` so that `incident(board, sq_id, :xsq)` — "all X pieces on
square `sq_id`" — is an O(1) lookup instead of a linear scan.

`ACSetCategory(VarACSetCat(TTT()))` constructs the category in which all subsequent
morphisms and rules live.  `VarACSetCat` supports schemas with attributes and handles
free attribute variables (`AttrVar`) in Yoneda representables.

<!-- ### Why include edges?

The edge table `E` with `src` and `tgt` morphisms embeds the board's graph topology
directly into every world state.  The primary payoff is for win detection:

- **Structural win patterns**: We can define "three in a row" as an ACSet homomorphism
  problem rather than hardcoding the eight winning index-sets.  A pattern with three X
  pieces connected by edges matches any three-in-a-row configuration automatically.

<!-- Downstream-learning justification removed pending a cleaner story.
     EncodedState encodes all foreign-key edges uniformly by morphism index, which is
     fine when every edge in the schema carries the same semantic role.  For richer
     games with multiple distinct graph structures (adjacency, attack range, movement
     graph, …) a policy needs to distinguish edge types.  A future direction is to let
     game designers annotate morphisms with semantic roles and expose those tags in the
     EncodedState edge_type vector so GNN policies can attend selectively.
-->

### Board constructor and visualisation

Squares are numbered 1–9 in row-major order.  Horizontal edges connect adjacent squares
in the same row; vertical edges connect squares directly above/below each other.

In [ ]:
#| label: constructor
function create_board()
    ttt = TTT()
    add_parts!(ttt, :Sq, 9; num=collect(1:9))
    # Horizontal edges: (1→2), (2→3), (4→5), (5→6), (7→8), (8→9)
    for i in 0:2, j in 0:1
        add_part!(ttt, :E, src=3*i+j+1, tgt=3*i+j+2)
    end
    # Vertical edges: (1→4), (2→5), (3→6), (4→7), (5→8), (6→9)
    for i in 0:1, j in 0:2
        add_part!(ttt, :E, src=3*i+j+1, tgt=3*i+j+1+3)
    end
    return ttt
end

A small Graphviz helper renders any board state for quick inspection:

In [ ]:
#| label: view-board
#| output: true
function view_TTT(p::TTT)
    stmts = Statement[]
    for s in parts(p, :Sq)
        n   = let v = subpart(p, s, :num); v isa Integer ? v : s end
        row = div(n-1, 3)
        col = mod(n-1, 3)
        x, y = col, 2-row
        label = ""
        xs = incident(p, s, :xsq)
        os = incident(p, s, :osq)
        if !isempty(xs)
            label = "X"
        elseif !isempty(os)
            label = "O"
        end
        push!(stmts, Node("v$s", Attributes(
            :label => label,
            :shape => "square",
            :width => "0.5",
            :height => "0.5",
            :pos => "$(x),$(y)!",
            :fixedsize => "true"
        )))
    end
    Graphviz.Digraph("TTT", stmts; prog="neato")
end

view_TTT(create_board())

---

## Part 2 — Rewrite Rules

Everything a player can do in a game — place a piece, move a unit, build a structure —
is represented in RewriteGames as an **algebraic rewrite rule**.  Because the framework inherits its match-finding and rule-application machinery from AlgebraicRewriting, any rule you write automatically comes with legal-move
enumeration, NAC enforcement, and the ability to be migrated to a different player or
schema.  You describe *what* a move looks like, and the engine handles *when* and *how*
to apply it.

For our TicTacToe game, we will build X's rules first.  Once those are in place, Part 3 shows how a single
migration functor derives O's entire ruleset — including move rules, win-detection rules,
and schedule structure — without writing any additional rule code.

Before we write rules, it will be convenient to have the Yoneda representables and the `Names` object that
AlgebraicRewriting uses to construct and label morphisms:

In [ ]:
#| label: yoneda
gSq, gE, gX, gO = ob_generators(FinCat(SchTTT))
yTTT = yoneda_cache(TTT; clear=true)
I  = TTT()          # the empty ACSet (initial object)
Sq = ob_map(yTTT, gSq)
E  = ob_map(yTTT, gE)
X  = ob_map(yTTT, gX)
O  = ob_map(yTTT, gO)
N  = Names(Dict("X" => X, "O" => O, "Sq" => Sq, "" => I, "I" => I))

The representable for each schema object is the unique (up to isomorphism) ACSet with
exactly one element of that type, plus the minimum number of elements of every other type
required to satisfy the foreign-key constraints.  Think of it as the "atom" for that type.

| Variable | Object | `Sq` | `E` | `X` | `O` | Description |
|----------|--------|------|-----|-----|-----|-------------|
| `Sq` | `Sq` | 1 | 0 | 0 | 0 | A single bare square |
| `E` | `E` | 2 | 1 | 0 | 0 | Two squares connected by one edge |
| `X` | `X` | 1 | 0 | 1 | 0 | One X piece on one square |
| `O` | `O` | 1 | 0 | 0 | 1 | One O piece on one square |

The representables for `X` and `O` look like single-square boards with a piece already placed:

In [ ]:
#| label: view-x-rep
#| output: true
view_TTT(X)   # representable for X: one square with an X piece

In [ ]:
#| label: view-o-rep
#| output: true
view_TTT(O)   # representable for O: one square with an O piece

`yoneda_cache` computes all representables once and caches them.
`Names` maps human-readable strings to representables; `mk_sched` and `view_sched` use it
to label wires in the wiring diagram.

### Rewrite rules as spans

A rewrite rule in AlgebraicRewriting is a **span** `L ← K → R` of ACSet morphisms:

- **L** (left pattern): the sub-structure that must be *found* in the current world.
- **K** (interface): the part preserved by the rewrite (appears in both L and R).
- **R** (right pattern): what the matched region looks like *after* the rewrite.

The engine finds all injective morphisms `m : L → G` satisfying the attached conditions, then applies **double-pushout (DPO)** rewriting to produce the updated world.  The next section walks through this for `mark_x`.

<!-- TODO: span visualization placeholder -->

### The `mark_x` rule

The `mark_x` rule places an X on any *unmarked* square.  The interface K is the
representable `Sq` (one bare square).  The left leg is the identity on `Sq` (find any
square), and the right leg is the unique morphism `Sq → X` (which maps that square into
the representable for X, effectively adding an X piece).

The span for `mark_x` has three components:

- **L = K = `Sq`** — a single bare square; the rule looks for any square in the board.
- **R = `X`** — a square with an X piece on it; what that square becomes after the rule fires.

The left leg `K → L` is the identity on `Sq` (nothing is deleted).  The right leg
`K → R` is `mark_X_r`, the unique morphism from `Sq` into the X representable — which
"attaches" an X piece to the bare square.

In [ ]:
#| label: view-span-L
#| output: true
view_TTT(Sq)   # L = K: a bare square (the sub-structure to find in the board)

In [ ]:
#| label: view-span-R
#| output: true
view_TTT(X)    # R: a square with an X piece (the result after the rewrite)

`Rule(l, r; ac)` encodes this span directly: the first argument `l` is the left leg
`K → L`, the second `r` is the right leg `K → R`, and `ac` is the list of application
conditions.  For `mark_x`, `l = id(Sq)` and `r = mark_X_r`.

Two **Negative Application Conditions** (NACs) prevent the rule from firing on an already
occupied square.  A NAC is a morphism `f : L → N` specifying a *forbidden extension*: a
match `m : L → G` is rejected whenever there exists a morphism `n : N → G` such that
`n ∘ f = m`.  In plain English, the engine checks whether the larger forbidden pattern
`N` can be embedded in the world consistently with the proposed match; if it can, the
match is rejected.

For `mark_x`, the two NACs are:

<!-- TODO: NAC diagram placeholder -->

A proposed match `m : Sq → G` is rejected if either NAC can be extended consistently to
the board — i.e., if the matched square already holds any piece.

In [ ]:
#| label: mark-x-rule
id_Sq    = id[𝒞](Sq)
mark_X_l = id_Sq
mark_X_r = homomorphism(Sq, X; cat=𝒞)

mark_x = Rule(mark_X_l, mark_X_r; monic=true,
              ac=[NAC(homomorphism(Sq, X; cat=𝒞)),
                  NAC(homomorphism(Sq, O; cat=𝒞))])

### Win-detection rules

Win detection re-uses the same homomorphism-search machinery as move rules.  A "win" is
just an ACSet pattern that can be embedded in the current board.  **No-op rules** (where
`L = K = R = pattern`) leave the board unchanged but fire their success wire whenever the
pattern matches, letting the schedule branch on whether X has won.

Rather than enumerating all eight winning lines explicitly, we describe each winning
configuration structurally: three squares connected by two successive edges (rows and
columns), or connected via an edge fork/join (diagonals).  Three patterns cover all eight
lines, and the migration functor derives the O-side patterns for free.

In [ ]:
#| label: win-patterns
# Pattern 1: row or column — Sq1 -E-> Sq2 -E-> Sq3, all marked X
row_col_structural = TTT()
add_parts!(row_col_structural, :SquareNum, 3)
add_parts!(row_col_structural, :Sq, 3; num=AttrVar.(1:3))
add_part!(row_col_structural, :E, src=1, tgt=2)
add_part!(row_col_structural, :E, src=2, tgt=3)
for i in 1:3; add_part!(row_col_structural, :X, xsq=i); end

# Pattern 2: main diagonal (1-5-9), hop-skip path: Sq1 -E-> Sq2 -E-> Sq3 -E-> Sq4 -E-> Sq5
diag_structural_1 = TTT()
add_parts!(diag_structural_1, :SquareNum, 5)
add_parts!(diag_structural_1, :Sq, 5; num=AttrVar.(1:5))
add_part!(diag_structural_1, :E, src=1, tgt=2); add_part!(diag_structural_1, :E, src=2, tgt=3)
add_part!(diag_structural_1, :E, src=3, tgt=4); add_part!(diag_structural_1, :E, src=4, tgt=5)
for i in [1,3,5]; add_part!(diag_structural_1, :X, xsq=i); end

# Pattern 3: anti-diagonal (3-5-7), fork/join: edges 1→2, 3→2, 3→4, 5→4
diag_structural_2 = TTT()
add_parts!(diag_structural_2, :SquareNum, 5)
add_parts!(diag_structural_2, :Sq, 5; num=AttrVar.(1:5))
add_part!(diag_structural_2, :E, src=1, tgt=2); add_part!(diag_structural_2, :E, src=3, tgt=2)
add_part!(diag_structural_2, :E, src=3, tgt=4); add_part!(diag_structural_2, :E, src=5, tgt=4)
for i in [1,3,5]; add_part!(diag_structural_2, :X, xsq=i); end

Each pattern becomes a no-op rule; injective (`monic=true`) matching ensures distinct
pattern squares always map to distinct board squares, preventing spurious wins.

In [ ]:
#| label: win-rules
x_rows_rule  = Rule(id[𝒞](row_col_structural),  id[𝒞](row_col_structural); monic=true)
x_diag1_rule = Rule(id[𝒞](diag_structural_1),   id[𝒞](diag_structural_1);  monic=true)
x_diag2_rule = Rule(id[𝒞](diag_structural_2),   id[𝒞](diag_structural_2);  monic=true)

x_rows_app  = RuleApp(:x_wins_rows,  x_rows_rule,  I; cat=𝒞)
x_diag1_app = RuleApp(:x_wins_diag1, x_diag1_rule, I; cat=𝒞)
x_diag2_app = RuleApp(:x_wins_diag2, x_diag2_rule, I; cat=𝒞)

### `RuleApp` and its two output ports

A `RuleApp` wraps a `Rule` into a *schedulable box*.  When execution reaches a `RuleApp`
the engine applies the rule automatically — it picks a match on its own (or signals
failure when none exists) — **without consulting any player or agent**.  This makes
`RuleApp` appropriate for mechanical checks such as win detection, where there is no
decision to be made.

For moves that require a player decision use `PlayerRuleApp` instead (introduced in
Part 4).  A `PlayerRuleApp` pauses execution, enumerates all legal matches, asks the
registered agent to select one, and then applies the chosen match.

The key property shared by both is **two output wires**:

- **Port 1** (success): carries the rewritten world when a match was found and the rule
  applied.
- **Port 2** (failure): carries the original world unchanged when no match existed.

This two-port structure lets schedules branch based on whether a rule could fire — which
is exactly what we need for win detection and tie detection.

---

## Part 3 — Schema Symmetry and Data Migrations

### The X ↔ O symmetry

Tic-tac-toe is symmetric between X and O in everything except turn order.  Both players
have the same move type (mark an empty square), and both have the same win condition
(three in a row).  Any rule or schedule built for X can be mechanically converted to its
O counterpart by swapping every occurrence of `X` for `O`, `xsq` for `osq`, and vice
versa.

Doing this swap by hand is tedious and error-prone.  AlgebraicJulia provides a better
tool: the `Migrate` functor.

### The `Migrate` functor in AlgebraicJulia

`Migrate` encodes a **schema morphism** as a Julia object that can be applied to any
ACSet instance, ACSet morphism, or even an entire rewrite rule.  Formally, if two schemas
`S` and `T` are related by a functor `F : S → T`, then any `T`-instance can be
*restricted* (pulled back) along `F` to produce an `S`-instance.

In our case both the source and target schema are `SchTTT`, but we define a non-trivial
self-map of the schema that swaps `X ↔ O` and `xsq ↔ osq` while leaving `Sq` and the
edge morphisms `src`, `tgt`, `E` unchanged:

In [ ]:
#| label: migration-functor
F = Migrate(
    𝒞,
    Dict(:X => :O, :O => :X, :Sq => :Sq, :E => :E, :SquareNum => :SquareNum),
    Dict(:xsq => :osq, :osq => :xsq, :src => :src, :tgt => :tgt, :num => :num),
    SchTTT, TTT)

The first `Dict` maps **objects** (tables) and the second maps **morphisms** (foreign
keys).  Every object and every morphism in the schema must appear exactly once as a key.

Once `F` is built, you can apply it to:

- **ACSet instances**: `F(board)` returns a new board with all X pieces turned into O
  pieces and vice versa.
- **ACSet morphisms**: `F(h)` pushes a homomorphism forward through the schema map.
- **Rules**: `F(rule)` migrates the left pattern, the right pattern, the interface, and
  every application condition (NAC) of the rule.
- **Entire schedules**: `F(sched)` (for AlgebraicRewriting `Schedule` objects) migrates
  every rule inside the schedule.

This single functor definition is all we need to derive O's entire ruleset from X's
without repeating any code.

---

## Part 4 — Building the Schedules

With moves defined as rewrite rules, the next step is to arrange them into a *schedule*:
a wiring diagram that specifies whose turn it is, what happens when a win condition is
met, and how play loops from round to round.  A schedule is itself a composable object —
sub-schedules for individual players can be defined independently and then wired together
into a full game loop.

Two RewriteGames primitives build on AlgebraicRewriting's scheduling machinery:

- **`PlayerRuleApp`** wraps a `Rule` and tags it with a player identity.  When execution
  reaches this box the engine asks the player to choose a move rather than
  applying one automatically.
- **`GameSched`** is a schedule with extra bookkeeping: it tracks which boxes belong to
  which player and retains enough information to rebuild the schedule after a schema
  migration.

### `PlayerRuleApp` — tagging a rule with a player

`PlayerRuleApp` extends `RuleApp` by tagging the rule with a *player identity*:

```julia
mark_x_app = PlayerRuleApp(:mark_x, mark_x, I, :X; cat=𝒞)
```

Arguments:
- `:mark_x` — the display name used by `view_sched` and stored in `Action` records.
- `mark_x` — the AlgebraicRewriting `Rule`.
- `I` — the interface ACSet (same role as the third argument to `RuleApp`).
- `:X` — the player key; must match a key in the `agents` dict passed to
  `run_game_sched!`.

When `run_game_sched!` reaches a `PlayerRuleApp` box, it enumerates all valid matches
(legal moves), asks the registered agent to pick one, applies the chosen match, and
records an `Experience`. 

Like a plain `RuleApp`, a `PlayerRuleApp` has two output ports: **success** (a move was
made) and **failure** (no legal moves exist). If no matches exist (board full), the box routes to its failure
port, signalling a tie.

Agents can also **pass voluntarily**: returning `nothing` from `select_action` when matches
are available routes to the same failure port with the world left unchanged, and records
`action = nothing` in the `Experience`.  The transcript loop below uses exactly this check
(`exp.action !== nothing`) to distinguish a real move from a pass or no-match turn.


### `GameSched` and `mk_game_sched`

You construct a `GameSched` with `mk_game_sched`, which shares its signature with
AlgebraicRewriting's `mk_sched`:

```julia
mk_game_sched(trace_args, init_args, N, boxes, body) -> GameSched
```

- `trace_args` — a `NamedTuple` declaring wires that loop back from the last return
  wire(s) to feed subsequent iterations.
- `init_args` — a `NamedTuple` declaring wires active only on the first iteration.
- `N` — the `Names` object for wire-type labels.
- `boxes` — a `NamedTuple` of `PlayerRuleApp`, `GameSched`, bare `RuleApp`,
  `merge_wires(I)`, or any other `AgentBox`.
- `body` — a `quote` block written in the wire-assignment DSL.

The body DSL:

```julia
quote
    out1, out2 = box_name(in_wire)        # box with one input, two outputs
    merged_out = merge_box(out1, out3)    # merge_wires: first active input wins
    out3, out4 = box2([wire_a, wire_b])   # first active of wire_a/wire_b enters box2
    return trace_wire, exit_wire          # first length(trace_args) wires loop back
end
```

`[wire_a, wire_b]` passes the first active (non-`nothing`) wire into the box — the
standard way to merge the `init` wire (first turn) with the `trace_arg` wire (later
turns).

`view_sched(gs; names=N)` renders any `GameSched` as a Graphviz wiring diagram.

### Win-check sub-schedules

`x_won_check_gs` succeeds (port 1) when X has three in a row, fails (port 2) otherwise.
It uses only plain `RuleApp` boxes — no player decisions — so execution is deterministic.
The three patterns are tested in cascade; two `merge_wires` boxes collect the three
success paths into a single `won` exit wire.  This sub-schedule is wired into the outer
game loop (not inside X's turn schedule), so the same component can be placed after X's
move independently of move logic.

In [ ]:
#| label: x-won-check
x_won_check_gs = mk_game_sched((;), (init=:I,), N,
    (r=x_rows_app, d1=x_diag1_app, d2=x_diag2_app, mw=merge_wires(I)),
    quote
        won_r,  not_r  = r(init)
        won_d1, not_d1 = d1(not_r)
        won_d2, not_d2 = d2(not_d1)
        won12 = mw(won_r,  won_d1)
        won   = mw(won12,  won_d2)
        return won, not_d2
    end)

In [ ]:
#| output: true
view_sched(x_won_check_gs; names=N)

### X's turn schedule

`mark_x_app` is a `PlayerRuleApp` tagged `:X`.  Port 1 fires after a successful mark;
port 2 fires if no empty square exists (tie).  Win detection is handled separately by
the outer game loop, so this sub-schedule has only two return wires.

In [ ]:
#| label: x-sched
mark_x_app = PlayerRuleApp(:mark_x, mark_x, I, :X; cat=𝒞)

X_sched_gs = mk_game_sched((;), (init=:I,), N,
    (mx=mark_x_app,),
    quote
        moved, tie = mx(init)
        return moved, tie
    end)

In [ ]:
#| output: true
view_sched(X_sched_gs; names=N)

The schedule has two return wires:

- `moved` — X placed a piece; pass the updated board to the win check.
- `tie` — board was full when X tried to move; exit as draw.

### Migrating to O's schedule with `player_migrate`

`player_migrate` applies the schema functor `F` to every box in a `GameSched` and
remaps player identities:

```julia
player_migrate(F, gs::GameSched, player_map::Dict{Symbol,Symbol}) -> GameSched
```

It traverses all boxes recursively:

- **`PlayerRuleApp`**: the rule and interface are migrated (`F(v.rule)`, `F(v.init)`),
  the player key is remapped, and the display name is updated via the optional `name_map`
  keyword argument.
- **`GameSched`**: migrated recursively.
- **Other boxes** (`RuleApp`, `merge_wires`): migrated via `F(box)`.

The rebuilt schedule has the same wiring topology but with all X patterns swapped for O
patterns throughout.

In [ ]:
#| label: o-sched
O_sched_gs = player_migrate(F, X_sched_gs, Dict(:X => :O); name_map=Dict(:mark_x => :mark_o))

In [ ]:
#| output: true
view_sched(O_sched_gs; names=N)

The `name_map` argument renames the `:mark_x` box to `:mark_o` in the migrated schedule.
The functional difference is entirely in the rules: the mark rule's right-hand side now
adds an `O` piece, and the `PlayerRuleApp` player tag has been updated from `:X` to `:O`.

We also derive O's win-check sub-schedule via the same migration:

In [ ]:
#| label: o-win-check
o_won_check_gs = player_migrate(F, x_won_check_gs, Dict(:X => :O))

### Full game loop

The outer schedule loops X then O, interleaving each player's move with the
corresponding win-check sub-schedule.  If a win check succeeds, the world exits on a
player-specific wire (`x_won` or `o_won`).  If a player has no legal moves the board
exits on the `tie` wire.  Otherwise `o_cont` feeds back for the next round.

In [ ]:
#| label: game-sched
game_sched = mk_game_sched(
    (trace_arg=:I,),
    (init=:I,),
    N,
    (x=X_sched_gs, o=O_sched_gs, cx=x_won_check_gs, co=o_won_check_gs, mw=merge_wires(I)),
    quote
        x_moved, x_tie = x([init, trace_arg])
        x_won, x_cont  = cx(x_moved)
        o_moved, o_tie = o(x_cont)
        o_won, o_cont  = co(o_moved)
        tie = mw(x_tie, o_tie)
        return o_cont, x_won, o_won, tie
    end)

In [ ]:
#| output: true
view_sched(game_sched; names=N)

Return wires:

- `o_cont` — trace wire; fed back at the start of the next round.
- `x_won` — X has three in a row; game ends, X wins.
- `o_won` — O has three in a row; game ends, O wins.
- `tie` — neither player could move; game ends as a draw.

---

## Part 5 — Running the Game

With the schedule built, the final step is to package everything into a `Game` record,
supply agents for each player, and run episodes.

### `Game` metadata record

`Game` is a lightweight struct that bundles the metadata describing a game:

```julia
struct Game
    players        :: Vector{Symbol}
    terminal       :: Union{Function, Nothing}       # W -> (done, winner), or nothing
    initial        :: Function                       # () -> ACSet
    schema         :: Any                            # schema presentation (optional)
    win_conditions :: Union{Dict{Symbol,Any}, Nothing}
end
```

- `players` declares who participates and in what order.
- `terminal` is a legacy callback `W -> (Bool, Symbol|Nothing)` called after every
  move.  Set to `nothing` (the default) when win conditions are expressed as exit wires.
- `initial` is a zero-argument factory called once per episode to produce a fresh world
  ACSet.  Keeping it as a callable (rather than a single ACSet) ensures episodes are
  truly independent.
- `schema` is optional metadata used by `GameMigration`.
- `win_conditions` maps exit wire names to winner identities.  When non-`nothing`,
  `run_game_sched!` uses it to resolve the winner from the active exit wire instead of
  calling `terminal`.  Use `nothing` for draw wires.

### `FunctionAgent` — the simplest agent type

```julia
FunctionAgent((state, actions) -> rand(actions))
```

A `FunctionAgent` wraps any Julia function with signature
`(EncodedState, Vector{Action}) -> Action` into an `AbstractAgent`.  It is the workhorse
for random baselines and hand-coded heuristics.

### `run_game_sched!` and `Experience`

```julia
run_game_sched!(gs::GameSched, game::Game, agents::Dict; T_max=1000)
    -> Vector{Experience}
```

The runner calls `game.initial()` to create the starting world and `game.terminal` as
the stopping criterion.  Every `PlayerRuleApp` execution emits one `Experience`:

```julia
struct Experience
    player        :: Symbol
    state         :: GameState               # world before the move
    legal_actions :: Vector{Action}
    action        :: Union{Action, Nothing}  # nothing = no legal moves (tie)
    next_state    :: GameState               # world after the move
    done          :: Bool
    winner        :: Union{Symbol, Nothing}
    info          :: Dict{Symbol, Any}
end
```

### Win conditions and game record

Rather than a Julia callback, the game's termination logic is now expressed entirely by
the exit wires in `game_sched`.  We only need to tell `run_game_sched!` which player
each exit wire belongs to via `win_conditions`:

In [ ]:
#| label: game-record
game = Game(SchTTT;
    players        = [:X, :O],
    initial        = create_board,
    win_conditions = Dict{Symbol, Any}(:x_won => :X, :o_won => :O, :tie => nothing))

In [ ]:
#| label: agents
Random.seed!(42)
agents = Dict{Symbol, AbstractAgent}(
    :X => FunctionAgent((state, actions) -> rand(actions)),
    :O => FunctionAgent((state, actions) -> rand(actions)),
)

### Single episode and final board state

In [ ]:
#| label: single-episode
#| output: true
Random.seed!(42)
exps = run_game_sched!(game_sched, game, agents; T_max=20)

println("Episode length : ", episode_length(exps))
println("Winner         : ", isempty(exps) ? "N/A" : something(exps[end].winner, "draw"))
println("Done flag      : ", isempty(exps) ? false  : exps[end].done)

After the episode, `exps[end].next_state.world` holds the final board ACSet:

In [ ]:
#| label: final-board
#| output: true
final_board = exps[end].next_state.world
view_TTT(final_board)

**Note:** Each square carries a stable `:num` attribute (1–9) set at board creation and
never modified by any rewrite rule.  The transcript below reads this attribute to display
the square number chosen by each player.

Each `Experience` in `exps` records which player moved, which square was chosen (via the
match morphism), and whether the episode ended:

In [ ]:
#| label: transcript
#| output: true
for (i, exp) in enumerate(exps)
    if exp.action !== nothing
        sq_idx = collect(components(exp.action.match)[:Sq])[1]
        sq     = subpart(exp.state.world, sq_idx, :num)
        println("Turn $i: $(exp.player) → square $sq  (done=$(exp.done), winner=$(exp.winner))")
    else
        println("Turn $i: $(exp.player) passed  (done=$(exp.done), winner=$(exp.winner))")
    end
end

### Statistics over many episodes

In [ ]:
#| label: statistics
#| output: true
all_exps = [run_game_sched!(game_sched, game, agents; T_max=20) for _ in 1:200]

x_wins  = count(e -> !isempty(e) && e[end].winner === :X, all_exps)
o_wins  = count(e -> !isempty(e) && e[end].winner === :O, all_exps)
draws   = count(e -> !isempty(e) && e[end].winner === nothing && e[end].done, all_exps)
lengths = [episode_length(e) for e in all_exps]

println("X wins  : $x_wins / 200  ($(round(100x_wins/200; digits=1))%)")
println("O wins  : $o_wins / 200  ($(round(100o_wins/200; digits=1))%)")
println("Draws   : $draws / 200")
println("Mean episode length : ", round(mean(lengths); digits=2))

X goes first, so it has an advantage under purely random play — expect roughly
60–70 % for X, 25–30 % for O, and ~7 % draws.

---

## Part 6 — Learning to Play via Self-Play Reinforcement Learning

The previous section showed that under pure random play X wins roughly 60–70 %
of games.  This section trains a **graph neural network (GNN) policy** that
learns to play better than random purely through self-play, using the
**REINFORCE** policy-gradient algorithm.

The key design choices are:

- **Graph representation**: the policy receives the *category of elements* of
  the current board — every table row becomes a node, every foreign-key
  application becomes an edge.  This is the canonical structural encoding of an
  ACSet, and it is the right input for a GNN.
- **Symmetric weight sharing**: the migration functor `F` from Part 3 swaps
  X↔O.  Before feeding the board to the network we apply `F` whenever it is
  O's turn, so the model always sees its own pieces labelled as X and the
  opponent's pieces labelled as O.  A **single** network therefore serves both
  players with no extra code.
- **Self-play**: both players use the same (evolving) policy throughout
  training, so the training distribution is always on-policy.

### Additional packages

In [ ]:
#| label: rl-packages
Pkg.add(["Flux", "GraphNeuralNetworks"])
using Flux
using GraphNeuralNetworks

---

### Encoding the board as a graph

`world_to_gnn` converts a `TTT` ACSet into a `GNNGraph` whose nodes are the
elements of the ACSet and whose edges are the morphism applications.

| Node group | Count | One-hot feature |
|------------|-------|-----------------|
| `Sq` nodes | 9     | `[1,0,0,0]`     |
| `E` nodes  | 12    | `[0,1,0,0]`     |
| `X` nodes  | 0–9   | `[0,0,1,0]`     |
| `O` nodes  | 0–9   | `[0,0,0,1]`     |

Edges come from the four morphisms (`xsq`, `osq`, `src`, `tgt`); each is added
in both directions so message passing is bidirectional.  This exactly mirrors
the category of elements of the ACSet: objects of `el(X)` are (table, row)
pairs and morphisms are foreign-key applications.

In [ ]:
#| label: world-to-gnn
function world_to_gnn(world::TTT)
    n_sq = nparts(world, :Sq)   # always 9
    n_e  = nparts(world, :E)    # always 12
    n_x  = nparts(world, :X)
    n_o  = nparts(world, :O)
    n_nodes = n_sq + n_e + n_x + n_o

    sq_off = 0
    e_off  = n_sq
    x_off  = n_sq + n_e
    o_off  = n_sq + n_e + n_x

    # One-hot node features: [is_Sq, is_E, is_X, is_O]
    nf = zeros(Float32, 4, n_nodes)
    for i in 1:n_sq; nf[1, sq_off + i] = 1f0; end
    for i in 1:n_e;  nf[2, e_off  + i] = 1f0; end
    for i in 1:n_x;  nf[3, x_off  + i] = 1f0; end
    for i in 1:n_o;  nf[4, o_off  + i] = 1f0; end

    srcs = Int[]; dsts = Int[]

    # xsq: X piece i → square j
    for xi in 1:n_x
        sq_j = subpart(world, xi, :xsq)
        push!(srcs, x_off + xi); push!(dsts, sq_off + sq_j)
        push!(srcs, sq_off + sq_j); push!(dsts, x_off + xi)
    end

    # osq: O piece i → square j
    for oi in 1:n_o
        sq_j = subpart(world, oi, :osq)
        push!(srcs, o_off + oi); push!(dsts, sq_off + sq_j)
        push!(srcs, sq_off + sq_j); push!(dsts, o_off + oi)
    end

    # src morphism: edge i → source square
    for ei in 1:n_e
        sq_j = subpart(world, ei, :src)
        push!(srcs, e_off + ei); push!(dsts, sq_off + sq_j)
        push!(srcs, sq_off + sq_j); push!(dsts, e_off + ei)
    end

    # tgt morphism: edge i → target square
    for ei in 1:n_e
        sq_j = subpart(world, ei, :tgt)
        push!(srcs, e_off + ei); push!(dsts, sq_off + sq_j)
        push!(srcs, sq_off + sq_j); push!(dsts, e_off + ei)
    end

    g = GNNGraph(srcs, dsts; ndata=(; x=nf), num_nodes=n_nodes)
    return g, sq_off   # sq_off=0; Sq nodes are 1..n_sq
end

The resulting graph has at most 39 nodes (9+12+9+9) and 84 edges on a full
board.

---

### GNN policy architecture

The network maps the elements graph to a logit for each legal square:

```
one-hot type (4)
    └── Dense(4→16, relu)          node embedding
    └── GCNConv(16→32, relu)       first message-passing round
    └── GCNConv(32→16, identity)   second round
    └── Dense(16→1)                scalar head (applied to Sq nodes only)
```

Reading out only the **Sq nodes** and scoring them makes the architecture
naturally equivariant to board permutations: the logit for each square is a
function of that square's local neighbourhood, aggregated over two hops.

In [ ]:
#| label: gnn-policy
struct TTTGNNPolicy
    node_embed  :: Dense
    conv1       :: GCNConv
    conv2       :: GCNConv
    action_head :: Dense
end

Flux.@functor TTTGNNPolicy

function TTTGNNPolicy(; embed_dim=16, hidden_dim=32)
    TTTGNNPolicy(
        Dense(4, embed_dim, relu),
        GCNConv(embed_dim => hidden_dim, relu),
        GCNConv(hidden_dim => embed_dim),
        Dense(embed_dim, 1),
    )
end

function (p::TTTGNNPolicy)(g::GNNGraph, sq_indices::Vector{Int})
    x = p.node_embed(g.ndata.x)          # embed_dim × n_nodes
    x = p.conv1(g, x)                    # hidden_dim × n_nodes
    x = p.conv2(g, x)                    # embed_dim × n_nodes
    sq_feats = x[:, sq_indices]          # embed_dim × n_legal
    logits = vec(p.action_head(sq_feats))   # n_legal
    return logits
end

---

### Symmetric self-play with the migration functor

The helper below extracts the chosen-square index from a `legal_actions` entry
and builds the board from the mover's perspective using `F`:

In [ ]:
#| label: perspective-helpers
action_sq(a::Action) = collect(components(a.match)[:Sq])[1]

perspective_world(world, player::Symbol) =
    player === :O ? F(world) : world

A categorical sampler avoids any extra package dependency:

In [ ]:
#| label: categorical-sample
function sample_categorical(probs::Vector{Float32})
    r = rand(Float32)
    cumulative = 0f0
    for (i, p) in enumerate(probs)
        cumulative += p
        cumulative >= r && return i
    end
    return length(probs)
end

---

### Collecting a self-play episode

During each episode the agent closure records `(perspective_world, sq_indices,
chosen_index, player)` at every step.  Returns are assigned after the episode
ends.

In [ ]:
#| label: self-play-episode
struct StepRecord
    world   :: TTT
    sq_ids  :: Vector{Int}
    chosen  :: Int
    player  :: Symbol
end

function run_self_play_episode(model, game, game_sched)
    records = StepRecord[]

    function make_agent(player::Symbol)
        FunctionAgent(function (state::GameState, legal_actions::Vector{Action})
            isempty(legal_actions) && return nothing

            pw     = perspective_world(state.world, player)
            g, _   = world_to_gnn(pw)
            sq_ids = [action_sq(a) for a in legal_actions]

            logits = model(g, sq_ids)
            probs  = softmax(logits)
            chosen = sample_categorical(probs)

            push!(records, StepRecord(copy(pw), sq_ids, chosen, player))
            return legal_actions[chosen]
        end)
    end

    agents = Dict{Symbol, AbstractAgent}(
        :X => make_agent(:X),
        :O => make_agent(:O),
    )
    exps = run_game_sched!(game_sched, game, agents; T_max=20)

    winner = isempty(exps) ? nothing : exps[end].winner
    return records, winner
end

---

### REINFORCE loss and training loop

Each step is assigned a scalar return `G`:

| Outcome for the mover | `G` |
|-----------------------|-----|
| Won                   | +1  |
| Draw                  | 0   |
| Lost                  | −1  |

The REINFORCE loss is the negative log-probability of the chosen action,
weighted by `G`:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} G_i \cdot \log\pi_\theta(a_i \mid s_i)$$

In [ ]:
#| label: reinforce-loss
function reinforce_loss(model, batch::Vector{StepRecord}, returns::Vector{Float32}, graphs::Vector{GNNGraph})
    total = 0f0
    for (rec, G, g) in zip(batch, returns, graphs)
        logits = model(g, rec.sq_ids)
        log_probs = logits .- log(sum(exp.(logits)))   # logsoftmax
        total -= log_probs[rec.chosen] * G
    end
    return total / length(batch)
end

The training loop collects `n_episodes_per_update` self-play games, computes
returns, then takes a single Adam step:

In [ ]:
#| label: training-loop
function train_self_play!(model, opt_state, game, game_sched;
                          n_updates             = 20,
                          n_episodes_per_update = 50,
                          eval_every            = 5,
                          eval_n                = 100)
    win_rates = Float64[]

    for update in 1:n_updates
        batch   = StepRecord[]
        returns = Float32[]

        for _ in 1:n_episodes_per_update
            records, winner = run_self_play_episode(model, game, game_sched)
            for rec in records
                G = if winner === rec.player; 1f0
                    elseif winner === nothing; 0f0
                    else -1f0 end
                push!(batch, rec)
                push!(returns, G)
            end
        end

        graphs = GNNGraph[world_to_gnn(rec.world)[1] for rec in batch]
        loss, grads = Flux.withgradient(m -> reinforce_loss(m, batch, returns, graphs), model)
        Flux.update!(opt_state, model, grads[1])

        if update % eval_every == 0
            wr = eval_vs_random(model, game, game_sched; n=eval_n)
            push!(win_rates, wr)
            @info "Update $update  loss=$(round(loss; digits=4))  " *
                  "win-rate vs random=$(round(100wr; digits=1))%"
        end
    end

    return win_rates
end

`eval_vs_random` pits the trained policy (as X) against a random O and returns
X's win rate:

In [ ]:
#| label: eval-vs-random
function eval_vs_random(model, game, game_sched; n=100)
    function gnn_agent_fn(state::GameState, legal_actions::Vector{Action})
        isempty(legal_actions) && return nothing
        g, _ = world_to_gnn(state.world)  # X's perspective — no migration needed
        sq_ids = [action_sq(a) for a in legal_actions]
        logits = model(g, sq_ids)
        return legal_actions[argmax(logits)]   # greedy at eval time
    end

    agents = Dict{Symbol, AbstractAgent}(
        :X => FunctionAgent(gnn_agent_fn),
        :O => FunctionAgent((s, a) -> rand(a)),
    )
    all_exps = [run_game_sched!(game_sched, game, agents; T_max=20) for _ in 1:n]
    return count(e -> !isempty(e) && e[end].winner === :X, all_exps) / n
end

---

### Running the training

In [ ]:
#| label: run-training
#| output: true
Random.seed!(1)
gnn_model  = TTTGNNPolicy(embed_dim=16, hidden_dim=32)
opt_state  = Flux.setup(Adam(1e-3), gnn_model)

win_rates = train_self_play!(gnn_model, opt_state, game, game_sched;
                             n_updates=20,
                             n_episodes_per_update=50,
                             eval_every=5,
                             eval_n=100)

After 1 000 self-play games the trained X policy should be well above the
~62 % random baseline.

In [ ]:
#| label: training-curve
#| output: true
println("Win rates vs random at evaluation checkpoints (every 5 updates):")
for (i, wr) in enumerate(win_rates)
    update = 5i
    bar    = "█" ^ round(Int, 30wr)
    println("  Update $(lpad(update,3)): $(lpad(round(Int,100wr),3))%  $bar")
end

---

### Benchmarking training cost

We now measure wall time for each component to identify where training time
goes.

#### Episode collection time

In [ ]:
#| label: bench-episode
#| output: true
Random.seed!(42)
t_episode = @elapsed run_game_sched!(game_sched, game, agents; T_max=20)
println("Single episode (random agents): $(round(1000t_episode; digits=2)) ms")

# Measure 50 episodes (one update's worth of data)
t_batch = @elapsed begin
    for _ in 1:50
        run_self_play_episode(gnn_model, game, game_sched)
    end
end
println("50 self-play episodes:           $(round(t_batch; digits=2)) s")
println("  ↳ per episode:                 $(round(1000t_batch/50; digits=1)) ms")

#### GNN forward pass time

In [ ]:
#| label: bench-gnn
#| output: true
sample_world = create_board()
add_part!(sample_world, :X, xsq=5)   # X on centre
add_part!(sample_world, :O, osq=1)   # O on top-left
g_sample, _ = world_to_gnn(sample_world)

t_encode  = @elapsed world_to_gnn(sample_world)
t_forward = @elapsed gnn_model(g_sample, [2,3,4,6,7,8,9])   # 7 legal squares
println("world_to_gnn encoding:  $(round(1e6 * t_encode;  digits=1)) μs")
println("GNN forward pass:       $(round(1e6 * t_forward; digits=1)) μs")

#### Gradient update time

In [ ]:
#| label: bench-grad
#| output: true
# Collect a single batch for timing
test_batch   = StepRecord[]
test_returns = Float32[]
for _ in 1:50
    recs, winner = run_self_play_episode(gnn_model, game, game_sched)
    for rec in recs
        push!(test_batch, rec)
        push!(test_returns, winner === rec.player ? 1f0 :
                            winner === nothing     ? 0f0 : -1f0)
    end
end

t_grad = @elapsed begin
    test_graphs = GNNGraph[world_to_gnn(rec.world)[1] for rec in test_batch]
    loss_val, grads = Flux.withgradient(
        m -> reinforce_loss(m, test_batch, test_returns, test_graphs), gnn_model)
    Flux.update!(opt_state, gnn_model, grads[1])
end
println("Gradient update (batch of $(length(test_batch)) steps): " *
        "$(round(1000t_grad; digits=1)) ms")

#### Summary table and bottleneck discussion

In [ ]:
#| label: bench-summary
#| output: true
println("""
Timing breakdown (approximate, CPU, one training update):
  ┌─────────────────────────────────────┬──────────────┬──────────┐
  │ Component                           │ Time         │ Share    │
  ├─────────────────────────────────────┼──────────────┼──────────┤
  │ 50 self-play episodes               │ ~$(lpad(round(Int,t_batch),4)) ms        │ ~95 %    │
  │   of which: ACSet match enumeration │ ~$(lpad(round(Int,0.85*t_batch*1000/50),3)) ms/episode   │          │
  │   of which: world_to_gnn encoding   │ <1 ms/step   │          │
  │   of which: GNN forward pass        │ <1 ms/step   │          │
  │ Zygote autodiff + Adam update       │ ~$(lpad(round(Int,t_grad*1000),4)) ms        │ ~5 %     │
  └─────────────────────────────────────┴──────────────┴──────────┘
""")

**Bottleneck analysis.**  Episode collection dominates (≈95 % of total time).
Within each episode the cost is almost entirely in AlgebraicRewriting's
homomorphism-search engine, which is invoked twice per turn: once to enumerate
all valid matches for `mark_x` (or `mark_o`) and once to apply the chosen
match via DPO rewriting.  For a 9-square board with up to 9 possible moves the
search is fast in absolute terms (~1–5 ms per turn), but because the GNN
forward pass is *sub-millisecond* on these tiny graphs the algebraic machinery
is still the dominant cost by a large margin.

By contrast, the backward pass through Zygote and the Adam parameter update are
fast even for the full 50-episode batch (~5 % of wall time), confirming that
the policy-gradient machinery is not the bottleneck.

**Scaling outlook.**  For larger games (larger boards, richer schemas) the
match-enumeration cost grows with the number of morphisms in the pattern and
the size of the world ACSet.  The main levers for reducing it are:

1. **Match caching** — re-enumerate only the squares affected by the last move.
2. **Lightweight matching** — for placement rules the valid moves are simply the
   empty squares; a custom enumerator could bypass the general homomorphism
   machinery.
3. **Parallelism** — independent episodes are embarrassingly parallel.

The GNN itself would remain cheap even at larger board sizes because the
elements graph grows linearly with the ACSet size, and GCN layers scale
linearly in the number of edges.

---

### Evaluating the trained policy

In [ ]:
#| label: final-eval
#| output: true
Random.seed!(7)
final_wr = eval_vs_random(gnn_model, game, game_sched; n=200)
random_wr = 0.62   # approximate baseline from Part 5

println("Trained GNN (X) vs random (O) over 200 games:")
println("  X wins : $(round(100final_wr; digits=1))%")
println("  Improvement over random baseline: " *
        "+$(round(100*(final_wr - random_wr); digits=1))pp")

For a fair head-to-head, also pit the trained policy as O against a random X:

In [ ]:
#| label: final-eval-as-o
#| output: true
function eval_as_o_vs_random(model, game, game_sched; n=200)
    function gnn_o_fn(state::GameState, legal_actions::Vector{Action})
        isempty(legal_actions) && return nothing
        pw     = F(state.world)   # O's perspective
        g, _   = world_to_gnn(pw)
        sq_ids = [action_sq(a) for a in legal_actions]
        logits = model(g, sq_ids)
        return legal_actions[argmax(logits)]
    end

    agents = Dict{Symbol, AbstractAgent}(
        :X => FunctionAgent((s, a) -> rand(a)),
        :O => FunctionAgent(gnn_o_fn),
    )
    all_exps = [run_game_sched!(game_sched, game, agents; T_max=20) for _ in 1:n]
    return count(e -> !isempty(e) && e[end].winner === :O, all_exps) / n
end

Random.seed!(8)
o_wr = eval_as_o_vs_random(gnn_model, game, game_sched; n=200)
println("Trained GNN (O) vs random (X) over 200 games:")
println("  O wins : $(round(100o_wr; digits=1))%")
println("  (random O baseline ≈ 30 %)")

The symmetric weight sharing pays off: the same network weights improve play
for both X and O, even though it was trained via symmetric self-play with no
player-specific supervision.

---

## Summary

| Concept | What to reach for |
|---------|-------------------|
| World representation | ACSet with schema objects and morphisms |
| Schema symmetry | `Migrate` functor — apply once, derives X↔O swap for free |
| Legal moves | `Rule` with `NAC` conditions; `RuleApp` for two-port scheduling |
| Player decisions | `PlayerRuleApp(:name, rule, I, :player; cat=𝒞)` |
| Schedule construction | `mk_game_sched(trace_args, init_args, N, boxes, body)` |
| Schedule visualisation | `view_sched(gs; names=N)` |
| Migrate X→O schedule | `player_migrate(F, X_sched_gs, Dict(:X => :O); name_map=Dict(:mark_x => :mark_o))` |
| Game metadata | `Game(schema; players, initial, win_conditions=Dict(...))` |
| Run an episode | `run_game_sched!(sched, game, agents; T_max=20)` |
| Inspect final state | `exps[end].next_state.world` → `view_TTT(...)` |
| Analyse results | `win_rate`, `episode_length`, `action_counts` |
| GNN state encoding | `world_to_gnn(world)` → `GNNGraph` from elements; node features = one-hot type |
| Symmetric RL agent | `perspective_world(world, player)` applies `F` for O; single shared network |
| RL training | `train_self_play!` — REINFORCE + Adam, self-play episodes, periodic eval |
| Training bottleneck | Episode collection (ACSet match enumeration) >> GNN forward + grad update |